In [51]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from xsml import data, features, splits, models, metrics, importance, libraries

## Valdez (2021) Cl-35 recreation 

This paper used KNN and DT regression, the former we have not explored yet. 

For a first run, we will use the techniques familiar to us already to try and replicate the cross-scetion prediction for Cl-35 (n,p) S-35


### What did he use?

The paper uses two main nuclear data sources - EXFOR and ENDF/B-VIII.0. They also used JENDL-4.0 for some parts

They also used about 4.5 million experimental data points from EXFOR , with a selection criteria:
- Only neutron-induced reaction cross sections
- Only data with MF=3
- Excluded cross sections that are reported as ratios or other derived quantities 
- Retained conflicting experimental campaigns rather than manually choosing which measurements were correct (avoids introducing human bias)

They did not use energy or cross section uncertainties to train models due to the scarcity of these in EXFOR 


### Features

| Feature                       | Description                       |
|-------------------------------|-----------------------------------|
| Incident Energy               | Neutron energy (eV)               |
| Z                             | Number of protons                 |
| N                             | Number of neutron                 |
| A                             | Mass number                       |
| MT                            | Reaction identifier               |
| CoM flag                      | Laboratory centre-of-mass data    |
| Target type                   | Isotopic or natural target        |
| Atomic mass                   | Atomic mass (Micro amu)           |
| Target atomic radius          | Nuclear radius (fm)               |
| Neutron / target radius ratio | Ratio of neutron and target radii |
| Mass excess                   | Nuclear mass excess               |
| Binding energy                | Nuclear binding energy            |
| Beta decay energy             | Beta minus decay energy           |
| S(2n)                         | Two-neutron separation energy     |
| S(2p)                         | Two-proton separation energy      |
| S(n)                          | Neutron separation energy         |
| S(p)                          | Proton separation energy          |


This feature set should be constructed from scratch as it is unique to this paper.


The data was split 80-10-10, training, validation and test, using a stratified technique over random to avoid over representing heavily measured isotopes. However, the exact splitting algorithm is not known therefore we must create one. 

## Getting the data ready

In [52]:
## Loading EXFOR data using xsml pipeline:

table = data.load()

display(table.head())
print("\n-----------------")
display(data.datasets(table).head())

,dataset,entry,subentry,author,year,reference,doi,entry_date,projectile,Z,...,reason,ds_flag,ds_offset_dex,ds_weight,pt_outlier,missing_uncertainty,relative_uncertainty_flag,uncertainty_unit_error,critique_kind,energy_frame
0,A0493008,A0493,A0493008,Neumaier,2002,"J,NP/A,712,247,2002",10.1016/S0375-9474(02)01274-5,2003-02-28,a,1,...,kept,unassessable,NaN,NaN,<NA>,False,False,False,,
1,O0744019,O0744,O0744019,Jaros,1978,"J,PR/C,18,2273,1978",10.1103/PhysRevC.18.2273,2000-03-29,a,1,...,kept,unassessable,NaN,NaN,<NA>,False,False,False,,
2,O0744019,O0744,O0744019,Jaros,1978,"J,PR/C,18,2273,1978",10.1103/PhysRevC.18.2273,2000-03-29,a,1,...,kept,unassessable,NaN,NaN,<NA>,False,False,False,,
3,O1011002,O1011,O1011002,De vismes,2002,"J,NP/A,706,295,2002",10.1016/S0375-9474(02)00753-4,2002-10-30,a,1,...,kept,unassessable,NaN,1.0,False,False,False,False,,
4,A1195002,A1195,A1195002,Bruno,1982,"J,NC/A,68,35,1982",10.1007/BF02902733,1984-02-17,a,1,...,kept,unassessable,NaN,1.0,False,False,False,False,,



-----------------


,dataset,entry,author,year,reference,entry_date,projectile,Z,A,MT,reaction,ds_flag,ds_offset_dex,ds_weight,points,e_min_eV,e_max_eV
0,10004002,10004,Schwartz,1974,"R,NBS-MONO-138,197401",1969-10-20,n,13,27,1,"13-AL-27(N,TOT),,SIG",ok,-0.001814,1.000000,3384,494600.0,16063900.0
1,10005003,10005,Schwartz,1969,"J,PL/B,30,36,196909",1969-09-11,n,1,1,1,"1-H-1(N,TOT),,SIG",ok,0.013602,0.987285,1652,1447000.0,14970000.0
2,10009002,10009,Whalen,1971,"P,ANL-7710,9,197101",1969-09-08,n,92,238,1,"92-U-238(N,TOT),,SIG",ok,-0.018907,0.989121,524,102200.0,1498100.0
3,10009003,10009,Whalen,1971,"P,ANL-7710,9,197101",1969-09-08,n,92,238,1,"92-U-238(N,TOT),,SIG",ok,-0.025412,0.999951,4,606670.0,1204100.0
4,10010002,10010,Smith,1970,"J,NSE,41,63,197007",1974-06-12,n,83,209,2,"83-BI-209(N,EL)83-BI-209,,SIG",ok,0.002943,1.000000,96,300000.0,1500000.0


Note that the paper was published in 2021, and states that results from (at the time) recent measurements (Batchelder et. al) were not included in the EXFOR training data, instead were used for validation. This means we must filter the original data accordingly, the date the paper was received was in April 2021, so this is a crude estimate. In reality, EXFOR data could still be more recent at this point and not included, e.g. they had already gathered all data and were writing up the paper. We should keep the Batchelder measurements completely separate until the end for external validation if we are to be true to the paper's methodology 

In [53]:
original_table = table.copy()

table["entry_date"] = pd.to_datetime(
    table["entry_date"].astype(str),
    errors="coerce"
)

table = table[
    (table["entry_date"] < "2021-04-23") &
    (~table["author"].str.contains("Batchelder", case=False, na=False))
].copy()

print(f"Latest remaining entry: {table["entry_date"].max()}")
print(
    "Remaining Batchelder records:",
    table["author"].str.contains("Batchelder", case=False, na=False).sum()
)

Latest remaining entry: 2021-03-31 00:00:00
Remaining Batchelder records: 0


In [54]:
table_n = data.select(table, projectile = "n", plain=True)
table_n = features.append(table_n)
table_n = table_n.reset_index(drop=True)                    ## Restart the table from 0
table_n["iso"] = data.isotope_id(table_n.Z, table_n.A)      ## Give isotope unique ID

print(f"Number of data points = {len(table_n)}")
print(f"Number of isotopes = {table_n.iso.nunique()}")

Number of data points = 3058958
Number of isotopes = 372


In [55]:
## Creating training and test splits

Z_target = 17
A_target = 35

## Data without chlorine isotopes

train_pool = table_n[
    ~((table_n["Z"] == Z_target) & (table_n["A"] == A_target))
].copy()

## Just chlorine data

cl = table_n[
    (table_n["Z"] == Z_target) & (table_n["A"] == A_target)
].copy()

print(f"Number of chlorine datapoints = {len(cl)}")
print(f"Number of datapoints viable for training = {len(train_pool)}")
print(f"Total data points = {len(cl) + len(train_pool)}")
print(f"Total data that it should be = {len(table_n)}")

Number of chlorine datapoints = 360
Number of datapoints viable for training = 3058598
Total data points = 3058958
Total data that it should be = 3058958


In [56]:
### Creating the split -- Try 80-10-10

folds = splits.isotope_folds(train_pool, n_folds=10)

train = train_pool[folds < 8].copy()
validation = train_pool[folds == 8].copy()
test = train_pool[folds == 9].copy()

print(f"Training:   {len(train):,} points ({len(train)/len(train_pool)*100:.1f}%)")
print(f"Validation: {len(validation):,} points ({len(validation)/len(train_pool)*100:.1f}%)")
print(f"Test:       {len(test):,} points ({len(test)/len(train_pool)*100:.1f}%)")

Training:   2,202,946 points (72.0%)
Validation: 239,354 points (7.8%)
Test:       616,298 points (20.1%)


We are splitting by isotope, so we won't get an exact 80:10:10 split, but this seems pretty close

In [57]:
## Try the no-leakage function to see if any isotopes appear in more than one fold:

splits.assert_no_leakage(train, validation, level="isotope")
splits.assert_no_leakage(train, test, level="isotope")
splits.assert_no_leakage(validation, test, level="isotope")

No error, so we are pretty confident we have split our data sufficiently

## Features...

In [58]:
table_n = data.select(table, projectile="n", plain=True)

table_n = features.append(table_n)              ## Adding AME parameters to dataset
table_n = table_n.reset_index(drop=True)

table_n["iso"] = data.isotope_id(table_n.Z, table_n.A)

print(table_n.columns.tolist())
print(table_n.dtypes)

['dataset', 'entry', 'subentry', 'author', 'year', 'reference', 'doi', 'entry_date', 'projectile', 'Z', 'A', 'MT', 'MF', 'reaction', 'product_state', 'sf5', 'sf8', 'sf9', 'energy_eV', 'sigma_b', 'unc_b', 'keep', 'reason', 'ds_flag', 'ds_offset_dex', 'ds_weight', 'pt_outlier', 'missing_uncertainty', 'relative_uncertainty_flag', 'uncertainty_unit_error', 'critique_kind', 'energy_frame', 'N', 'element', 'mass_excess', 'mass_excess_unc', 'estimated', 'atomic_mass_u', 'binding_energy', 'B_per_A', 'S_n', 'S_p', 'S_2n', 'S_2p', 'Q_alpha', 'Q_beta_minus', 'Q_ng', 'Q_np', 'Q_na', 'Q_n2n', 'Q_nd', 'Q_nt', 'N_minus_Z', 'asymmetry', 'Z_odd', 'N_odd', 'A_odd', 'pairing', 'Z_magic_dist', 'N_magic_dist', 'Z_magic', 'N_magic', 'P_factor', 'A_third', 'radius_fm', 'log10_energy', 'log10_sigma', 'Q_reaction', 'iso']
dataset         category
entry           category
subentry        category
author          category
year               Int16
                  ...   
radius_fm        float64
log10_energy    

There are some features we need to exclude such as metadata or data-quality flags that are not useful to us. Also it is very important we remove log10(sigma) because that is what we are trying to predict. There are slight differences in the way that the features are listed in the paper vs what we have, e.g. the xsml toolkit uses MeV for nuclear energy quantities, whilst the paper uses keV. 

There is some general feature mapping that needs to be done as our features currently are not exactly the same as in the paper

We do not have the necessary information to construct the following parameters:
- Centre-of-mass flag
- Target type


And the neutron/target radius ratio was assumed to be calculated as $0.8 / R_{target}$

In [59]:
table_n["radius_ratio"] = 0.8 / table_n["radius_fm"]
train_pool["radius_ratio"] = 0.8 / train_pool["radius_fm"]

## Constructing features 

feature_cols = [
    "energy_eV",
    "Z",
    "N",
    "A",
    "MT",
    "atomic_mass_u",
    "radius_fm",
    "radius_ratio",
    "mass_excess",
    "binding_energy",
    "Q_beta_minus",
    "S_2n",
    "S_2p",
    "S_n",
    "S_p"
]


In [60]:
## Input and target features:

X = train_pool[feature_cols].copy()
y = train_pool["sigma_b"].copy()

# Checking for missing data:
print("Missing feature values:")
print(X.isna().sum())

print("\nInfinite feature values:")
print(np.isinf(X.select_dtypes(include="number")).sum())

print("\nMissing target values:", y.isna().sum())
print("Non-positive target values:", (y <= 0).sum())

Missing feature values:
energy_eV             0
Z                     0
N                     0
A                     0
MT                    0
atomic_mass_u      5302
radius_fm          5302
radius_ratio       5302
mass_excess        5302
binding_energy     5302
Q_beta_minus       6595
S_2n              12524
S_2p               6922
S_n               11102
S_p                5500
dtype: int64

Infinite feature values:
energy_eV         0
Z                 0
N                 0
A                 0
MT                0
atomic_mass_u     0
radius_fm         0
radius_ratio      0
mass_excess       0
binding_energy    0
Q_beta_minus      0
S_2n              0
S_2p              0
S_n               0
S_p               0
dtype: int64

Missing target values: 0
Non-positive target values: 0


Of course, there are some missing data, so we need to see how bad it is and whether we can just clean it...

In [61]:
# Number of rows before cleaning
print("Before:", len(train_pool))

# Remove rows with missing input features
train_clean = train_pool.dropna(subset=feature_cols).copy()

print("After:", len(train_clean))
print("Removed:", len(train_pool) - len(train_clean))
print("Percentage retained:",
      100 * len(train_clean) / len(train_pool))

Before: 3058598


After: 3045876
Removed: 12722
Percentage retained: 99.58405779379964


This isn't actually that bad, less than 0.5% of the data was stripped

In [62]:
cl["radius_ratio"] = 0.8 / cl["radius_fm"]
cl_clean = cl.dropna(subset=feature_cols).copy()

print("Chlorine rows before:", len(cl))
print("Chlorine rows after:", len(cl_clean))

Chlorine rows before: 360
Chlorine rows after: 360


Chlorine seems to be okay

In [63]:
# Add radius ratio to each split
for df in [train, validation, test, cl]:
    df["radius_ratio"] = 0.8 / df["radius_fm"]


### REDOING THE CLEANING

# Clean each dataset independently
train_clean = train.dropna(subset=feature_cols).copy()
validation_clean = validation.dropna(subset=feature_cols).copy()
test_clean = test.dropna(subset=feature_cols).copy()
cl_clean = cl.dropna(subset=feature_cols).copy()

# Define inputs and targets
X_train = train_clean[feature_cols].copy()
y_train = train_clean["sigma_b"].copy()

X_val = validation_clean[feature_cols].copy()
y_val = validation_clean["sigma_b"].copy()

X_test = test_clean[feature_cols].copy()
y_test = test_clean["sigma_b"].copy()

X_cl = cl_clean[feature_cols].copy()
y_cl = cl_clean["sigma_b"].copy()

The paper treats MT as a categorical feature therefore needs one-hot encoding

In [64]:
# Only encode features treated as categorical in the paper
categorical_cols = ["MT"]

# Convert categorical columns to strings
for df in [X_train, X_val, X_test, X_cl]:
    for col in categorical_cols:
        df[col] = df[col].astype(str)

# One-hot encode each dataset
X_train = pd.get_dummies(X_train, columns=categorical_cols)
X_val = pd.get_dummies(X_val, columns=categorical_cols)
X_test = pd.get_dummies(X_test, columns=categorical_cols)
X_cl = pd.get_dummies(X_cl, columns=categorical_cols)

# Ensure all datasets have the same feature columns as training
X_val = X_val.reindex(columns=X_train.columns, fill_value=0)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)
X_cl = X_cl.reindex(columns=X_train.columns, fill_value=0)

In [69]:
y_train = np.log10(train_clean["sigma_b"])
y_val = np.log10(validation_clean["sigma_b"])
y_test = np.log10(test_clean["sigma_b"])
y_cl = np.log10(cl_clean["sigma_b"])

for name, df in [
    ("Train", train_clean),
    ("Validation", validation_clean),
    ("Test", test_clean),
    ("Chlorine", cl_clean)
]:
    print(name, (df["sigma_b"] <= 0).sum())

Train 0
Validation 0
Test 0
Chlorine 0


### Training

In [70]:
from sklearn.tree import DecisionTreeRegressor

# Initialise the model
dt_model = DecisionTreeRegressor(random_state=0)

# Train the model
dt_model.fit(X_train, y_train)

,criterion,'squared_error'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,ccp_alpha,0.0


In [67]:
# Predictions
y_train_pred = dt_model.predict(X_train)
y_val_pred = dt_model.predict(X_val)

## Other test

In [65]:
display(
    cl_np.groupby(
        ["dataset", "author", "year"],
        observed=True
    )
    .agg(
        points=("sigma_b", "size"),
        min_energy=("energy_eV", "min"),
        max_energy=("energy_eV", "max")
    )
    .reset_index()
)

NameError: name 'cl_np' is not defined

We see that the majority of Cl measurements are from 2 or 3 experimental campaigns, the rest just give a few datapoints.

The paper was published in 2021, so that rules out the Hanselman 2024 data, and the Batchelder data from UCB was used exclusively for validation so we should follow suit

In [ ]:
display(
    cl_np[
        ["dataset", "author", "year", "energy_eV", "sigma_b"]
    ].sort_values("energy_eV")
)

,dataset,author,year,energy_eV,sigma_b
441131,10042002,Sims,1969,2.530000e-02,0.48900
441132,13533002,Koehler,1991,2.750000e-02,0.47360
441133,13533002,Koehler,1991,3.130000e-02,0.43620
441134,13533002,Koehler,1991,3.560000e-02,0.42500
441135,13533002,Koehler,1991,4.040000e-02,0.38940
...,...,...,...,...,...
441677,14840006,Hanselman,2024,7.062000e+06,0.19092
441681,40433009,Aleksandrov,1975,1.410000e+07,0.33000
441678,20198006,Nagel,1966,1.460000e+07,0.11700
441679,20198007,Nagel,1966,1.460000e+07,0.13100


In [ ]:
## The Batchelder data...

batchelder = cl_np[cl_np["author"] == "Batchelder"].copy()

display(
    batchelder[
        [
            "dataset", "entry", "subentry",
            "author", "year", "reference", "doi",
            "energy_eV", "sigma_b", "unc_b"
        ]
    ].sort_values("energy_eV")
)

display(
    batchelder[
        ["dataset", "entry", "subentry", "reference", "doi"]
    ].drop_duplicates()
)

,dataset,entry,subentry,author,year,reference,doi,energy_eV,sigma_b,unc_b
441300,145560031,14556,14556003,Batchelder,2019,"J,PR/C,99,044612,2019",10.1103/PhysRevC.99.044612,2420000.0,0.0166,0.005322
441305,145560032,14556,14556003,Batchelder,2019,"J,PR/C,99,044612,2019",10.1103/PhysRevC.99.044612,2420000.0,0.0196,0.005430
441301,145560031,14556,14556003,Batchelder,2019,"J,PR/C,99,044612,2019",10.1103/PhysRevC.99.044612,2520000.0,0.0261,0.003582
441306,145560032,14556,14556003,Batchelder,2019,"J,PR/C,99,044612,2019",10.1103/PhysRevC.99.044612,2520000.0,0.0257,0.003677
441302,145560031,14556,14556003,Batchelder,2019,"J,PR/C,99,044612,2019",10.1103/PhysRevC.99.044612,2580000.0,0.0446,0.003270
441307,145560032,14556,14556003,Batchelder,2019,"J,PR/C,99,044612,2019",10.1103/PhysRevC.99.044612,2580000.0,0.0417,0.004662
441303,145560031,14556,14556003,Batchelder,2019,"J,PR/C,99,044612,2019",10.1103/PhysRevC.99.044612,2640000.0,0.0500,0.003884
441308,145560032,14556,14556003,Batchelder,2019,"J,PR/C,99,044612,2019",10.1103/PhysRevC.99.044612,2640000.0,0.0478,0.004048
441298,14556002,14556,14556002,Batchelder,2019,"J,PR/C,99,044612,2019",10.1103/PhysRevC.99.044612,2740000.0,0.0284,0.001957
441299,14556002,14556,14556002,Batchelder,2019,"J,PR/C,99,044612,2019",10.1103/PhysRevC.99.044612,2740000.0,0.0281,0.001887


,dataset,entry,subentry,reference,doi
441298,14556002,14556,14556002,"J,PR/C,99,044612,2019",10.1103/PhysRevC.99.044612
441300,145560031,14556,14556003,"J,PR/C,99,044612,2019",10.1103/PhysRevC.99.044612
441305,145560032,14556,14556003,"J,PR/C,99,044612,2019",10.1103/PhysRevC.99.044612


We confirm that these are indeed the same Batcheder measurements that were used in the paper (All have the same reference and DOI)

We should note that the data in the xsml pipeline is more recent than the data that was used in the original paper, so we may need to sort by this as well. The year 2019 seems like the cutoff, the paper mentions not having access to the Batchelder data when the models were trained, so 2019 seems suitable. 